# Занятие 8. Лабораторная: свое дерево решений

Срок сдачи — 31 октября 2026 года, ноутбук присылают преподавателю.
В лабораторной двенадцать задач: девять обязательных и три бонусные,
у бонусных это написано в заголовке. Обязательные задачи дают до 10 баллов;
задачи 5 и 6 весят вдвое больше остальных, задача 7 — в полтора раза.
Каждая бонусная задача дает до балла сверху пропорционально доле пройденных
проверок; итог округляется вниз и не больше 11.

Цель — написать дерево решений целиком, без sklearn: критерии
информативности, поиск лучшего разбиения, рекурсивное построение,
классы с интерфейсом `fit` и `predict` для классификации и регрессии,
важность признаков. Затем свое дерево сравнивается с `DecisionTreeClassifier`
по качеству и времени. В бонусах — быстрый поиск порога, отсечение
по сложности и предсказание при пропущенных значениях.

**Заготовки.** В каждой заготовке указано, что должно получиться: тип
и форма в аннотации, размеры входов и выходов в комментарии перед функцией.
Как считать — в теории перед задачей, в семинаре и в конспекте занятия.
Имена переменных и функций менять нельзя, проверка ищет их по именам.
Вспомогательные функции можно добавлять свои. После каждой задачи идет
ячейка с открытыми проверками; при сдаче работа дополнительно проверяется
закрытыми тестами на других данных.

**Что нельзя использовать.** В функциях и классах задач 1–7, 9 и бонусов
нельзя вызывать деревья и ансамбли sklearn; numpy использовать можно.
sklearn нужен только в проверках и для сравнения в задачах 8 и 9.

**Данные.** Вина (`load_wine`) для классификации и диабет (`load_diabetes`)
для регрессии, как на семинаре. Метки классов — целые числа от 0 до $K - 1$.

In [ ]:
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.axisbelow": True,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
SEED = 42

wine = load_wine()
FEATURE_NAMES = list(wine.feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.3, random_state=SEED, stratify=wine.target
)

diabetes = load_diabetes()

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.3, random_state=SEED
)

print("вина:", X_train.shape, X_test.shape)
print("диабет:", Xd_train.shape, Xd_test.shape)

## Блок A. Критерии и разбиение

$p_k$ — доля класса $k$ среди объектов узла $R$. Индекс Джини и энтропия в битах:

$$H_{\text{Gini}}(R) = 1 - \sum_{k=0}^{K-1} p_k^2, \qquad H_{\text{ent}}(R) = -\sum_{k:\, p_k > 0} p_k \log_2 p_k.$$

### Задача 1. Критерии информативности

Функции `gini` и `entropy` получают вектор меток узла и число классов $K$.
Классы, которых нет в узле, вклада не дают.

In [ ]:
# y: (n,) целые метки от 0 до n_classes - 1, n >= 1
# возвращает число: индекс Джини узла
def gini(y: np.ndarray, n_classes: int) -> float:
    return ...


# y: (n,) целые метки от 0 до n_classes - 1, n >= 1
# возвращает число: энтропия узла в битах
def entropy(y: np.ndarray, n_classes: int) -> float:
    return ...

In [ ]:
# --- проверка ---
two_equal_classes = np.array([0, 0, 1, 1])
assert np.isclose(gini(two_equal_classes, 2), 0.5), "два равных класса: индекс Джини 0.5"
assert np.isclose(entropy(two_equal_classes, 2), 1.0), "два равных класса: энтропия 1 бит"

pure_node = np.array([2, 2, 2])
assert np.isclose(gini(pure_node, 3), 0.0), "чистый узел: индекс Джини 0"
assert np.isclose(entropy(pure_node, 3), 0.0), "чистый узел: энтропия 0"

three_equal_classes = np.array([0, 1, 2])
assert np.isclose(gini(three_equal_classes, 3), 2 / 3), "три равных класса: индекс Джини 2/3"
assert np.isclose(entropy(three_equal_classes, 3), np.log2(3)), "три равных класса: энтропия log2(3)"

assert np.isfinite(entropy(np.array([0, 0, 1]), 4)), "отсутствующие классы не должны давать nan"

gini_root = gini(y_train, 3)
entropy_root = entropy(y_train, 3)
assert np.isclose(gini_root, 0.6573, atol=1e-4), "корень вин: индекс Джини 0.657, как на семинаре"
print(f"корень вин: Джини {gini_root:.4f}, энтропия {entropy_root:.4f} бита")
print("проверки пройдены")

Словарь критериев, через который следующие функции получают критерий по имени:

In [ ]:
CRITERIA = {"gini": gini, "entropy": entropy}

### Задача 2. Прирост информации

Разбиение делит объекты узла $R$ на левого потомка $R_l$ (маска `left_mask`)
и правого $R_r$. Прирост информации:

$$Q = H(R) - \frac{|R_l|}{|R|} H(R_l) - \frac{|R_r|}{|R|} H(R_r).$$

In [ ]:
# y: (n,) метки, left_mask: (n,) bool — True у объектов левого потомка; оба потомка непустые
# criterion: "gini" или "entropy"
# возвращает число: прирост информации Q
def information_gain(y: np.ndarray, left_mask: np.ndarray, criterion: str, n_classes: int) -> float:
    return ...

In [ ]:
# --- проверка ---
y_toy = np.array([0] * 400 + [1] * 400)          # объекты 0–399 класса 0, объекты 400–799 класса 1

# разбиение A из конспекта: влево 300 объектов класса 0 и 100 класса 1 — потомки (300, 100) и (100, 300)
mask_a = np.zeros(800, dtype=bool)
mask_a[:300] = True
mask_a[400:500] = True

# разбиение B: влево 200 объектов класса 0 и все 400 класса 1 — потомки (200, 400) и (200, 0)
mask_b = np.zeros(800, dtype=bool)
mask_b[:200] = True
mask_b[400:] = True

assert np.isclose(information_gain(y_toy, mask_a, "gini", 2), 0.125), "разбиение A, индекс Джини: 0.125"
assert np.isclose(information_gain(y_toy, mask_b, "gini", 2), 1 / 6), "разбиение B, индекс Джини: 0.1667"
assert np.isclose(information_gain(y_toy, mask_a, "entropy", 2), 0.1887, atol=1e-4), "разбиение A, энтропия: 0.1887"
assert np.isclose(information_gain(y_toy, mask_b, "entropy", 2), 0.3113, atol=1e-4), "разбиение B, энтропия: 0.3113"

gain_a = information_gain(y_toy, mask_a, "gini", 2)
gain_a_swapped = information_gain(y_toy, ~mask_a, "gini", 2)
assert np.isclose(gain_a, gain_a_swapped), "перестановка потомков не меняет прирост"

color_column = FEATURE_NAMES.index("color_intensity")
root_mask = X_train[:, color_column] <= 3.82
root_gain = information_gain(y_train, root_mask, "gini", 3)
print(f"корень вин, вопрос color_intensity <= 3.82: прирост по Джини {root_gain:.4f}")
print("проверки пройдены")

### Задача 3. Лучший порог по одному признаку

Пороги-кандидаты — середины между соседними различными значениями признака
в узле. Порог допустим, если в каждом потомке не меньше `min_samples_leaf`
объектов. Функция возвращает допустимый порог с наибольшим приростом
и сам прирост; при равном приросте — наименьший порог. Если допустимых
порогов нет (все значения признака одинаковы или потомки слишком малы),
функция возвращает `None`. Объект идет влево, если $x \le t$.

In [ ]:
# x: (n,) значения одного признака у объектов узла, y: (n,) метки
# возвращает (порог, прирост) или None
def best_threshold(x: np.ndarray, y: np.ndarray, criterion: str, n_classes: int,
                   min_samples_leaf: int = 1) -> tuple[float, float] | None:
    return ...

In [ ]:
# --- проверка ---
x_toy = np.array([1.0, 2.0, 3.0, 4.0])
y_toy = np.array([0, 0, 1, 1])

threshold, gain = best_threshold(x_toy, y_toy, "gini", 2)
assert np.isclose(threshold, 2.5), "x = 1, 2, 3, 4 и метки 0, 0, 1, 1: порог 2.5"
assert np.isclose(gain, 0.5), "x = 1, 2, 3, 4 и метки 0, 0, 1, 1: прирост 0.5"

assert best_threshold(x_toy, y_toy, "gini", 2, min_samples_leaf=3) is None, "при min_samples_leaf = 3 из четырех объектов разбить нельзя"

constant_x = np.full(5, 7.0)
assert best_threshold(constant_x, np.array([0, 1, 0, 1, 0]), "gini", 2) is None, "у постоянного признака нет порогов"

x_with_repeats = np.array([1.0, 1.0, 2.0, 2.0, 3.0])
threshold, gain = best_threshold(x_with_repeats, np.array([0, 0, 1, 1, 1]), "entropy", 2)
assert np.isclose(threshold, 1.5), "повторяющиеся значения: пороги только между различными значениями"

color_column = FEATURE_NAMES.index("color_intensity")
stump = DecisionTreeClassifier(max_depth=1)
stump.fit(X_train[:, [color_column]], y_train)
sklearn_threshold = stump.tree_.threshold[0]

threshold, gain = best_threshold(X_train[:, color_column], y_train, "gini", 3)
assert np.isclose(threshold, sklearn_threshold, atol=1e-5), f"порог {threshold:.4f} не совпал с sklearn {sklearn_threshold:.4f}"
print(f"color_intensity: порог {threshold:.3f}, прирост {gain:.4f}")
print("проверки пройдены")

### Задача 4. Лучшее разбиение узла

Функция `best_split` перебирает все признаки и возвращает номер признака,
порог и прирост лучшего допустимого разбиения; при равном приросте —
признак с меньшим номером. Если ни у одного признака нет допустимого порога,
функция возвращает `None`.

In [ ]:
# X: (n, p) объекты узла, y: (n,) метки
# возвращает (номер признака, порог, прирост) или None
def best_split(X: np.ndarray, y: np.ndarray, criterion: str, n_classes: int,
               min_samples_leaf: int = 1) -> tuple[int, float, float] | None:
    return ...

In [ ]:
# --- проверка ---
feature, threshold, gain = best_split(X_train, y_train, "gini", 3)

stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
stump.fit(X_train, y_train)
structure = stump.tree_                       # узлы: 0 — корень, 1 — левый лист, 2 — правый лист

assert feature == structure.feature[0], "признак в корне вин должен совпасть с sklearn"
assert np.isclose(threshold, structure.threshold[0], atol=1e-5), "порог в корне вин должен совпасть с sklearn"

node_share = structure.weighted_n_node_samples / structure.weighted_n_node_samples[0]
sklearn_gain = structure.impurity[0] - node_share[1] * structure.impurity[1] - node_share[2] * structure.impurity[2]
assert np.isclose(gain, sklearn_gain), "прирост в корне не совпал с пересчетом по дереву sklearn"

constant_X = np.ones((6, 3))
assert best_split(constant_X, np.array([0, 1, 0, 1, 0, 1]), "gini", 2) is None, "все признаки постоянны — разбиения нет"
print(f"корень вин: {FEATURE_NAMES[feature]} <= {threshold:.3f}, прирост {gain:.4f}")
print("проверки пройдены")

## Блок B. Дерево

Узел дерева — словарь. У каждого узла есть ключи:

- `"value"` — `np.ndarray` формы `(n_classes,)`, доли классов среди обучающих
  объектов узла;
- `"n_samples"` — `int`, число обучающих объектов узла;
- `"impurity"` — `float`, значение критерия на объектах узла.

У внутреннего узла дополнительно:

- `"feature"` — `int`, номер признака; `"threshold"` — `float`, порог;
- `"gain"` — `float`, прирост информации разбиения;
- `"left"` — узел для объектов с `x[feature] <= threshold`, `"right"` — для остальных.

У листа ключей `"feature"`, `"threshold"`, `"gain"`, `"left"`, `"right"` нет.

Узел становится листом, если выполнено хотя бы одно условие: все объекты
узла одного класса; глубина узла равна `max_depth` (у корня глубина 0;
`max_depth=None` — без ограничения); объектов в узле меньше `min_samples_split`;
`best_split` вернул `None`. Иначе узел делят лучшим разбиением,
даже если его прирост равен нулю, — так же поступает sklearn.

### Задача 5. Построение дерева

In [ ]:
# X: (n, p), y: (n,) метки от 0 до n_classes - 1; depth — глубина текущего узла, у корня 0
# возвращает корень дерева — узел-словарь в формате из условия
def build_tree(X: np.ndarray, y: np.ndarray, criterion: str, n_classes: int, max_depth: int | None = None,
               min_samples_split: int = 2, min_samples_leaf: int = 1, depth: int = 0) -> dict:
    return ...

In [ ]:
# --- проверка ---
def leaves_of(node):
    found = [node] if "feature" not in node else leaves_of(node["left"]) + leaves_of(node["right"])
    return found


tree_2 = build_tree(X_train, y_train, "gini", 3, max_depth=2)
leaves_2 = leaves_of(tree_2)
root_shares = np.bincount(y_train) / len(y_train)

assert tree_2["feature"] == FEATURE_NAMES.index("color_intensity"), "корень: вопрос по color_intensity"
assert np.isclose(tree_2["threshold"], 3.82, atol=0.01), "корень: порог 3.82"
assert tree_2["n_samples"] == len(y_train), "корень хранит число обучающих объектов"
assert np.allclose(tree_2["value"], root_shares), "корень хранит доли классов"

assert len(leaves_2) == 4, "у дерева глубины 2 на винах четыре листа"
assert sum(leaf["n_samples"] for leaf in leaves_2) == len(y_train), "листья делят все обучающие объекты"
assert all(np.isclose(leaf["value"].sum(), 1) for leaf in leaves_2), "доли классов листа в сумме дают 1"

full_tree = build_tree(X_train, y_train, "gini", 3)
full_leaves = leaves_of(full_tree)
assert all(leaf["impurity"] == 0 for leaf in full_leaves), "дерево без ограничений строится до чистых листьев"

tree_min_leaf = build_tree(X_train, y_train, "entropy", 3, min_samples_leaf=5)
smallest_leaf = min(leaf["n_samples"] for leaf in leaves_of(tree_min_leaf))
assert smallest_leaf >= 5, "min_samples_leaf соблюден"

print(f"листьев: глубина 2 — {len(leaves_2)}, без ограничений — {len(full_leaves)}")
print("проверки пройдены")

### Задача 6. Классификатор с интерфейсом sklearn

Класс `MyDecisionTreeClassifier` строит дерево функцией `build_tree`
и хранит корень в атрибуте `tree_`, число классов — в `n_classes_`
(наибольшая метка плюс один). `predict_proba` возвращает доли классов листа,
в который попал объект, `predict` — класс с наибольшей долей (при равенстве —
с меньшим номером). `get_depth` — наибольшая глубина листа, `get_n_leaves` —
число листьев.

In [ ]:
class MyDecisionTreeClassifier:
    def __init__(self, criterion: str = "gini", max_depth: int | None = None,
                 min_samples_split: int = 2, min_samples_leaf: int = 1):
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf

    # X: (n, p), y: (n,) метки от 0 до K - 1; задает self.n_classes_ и self.tree_, возвращает self
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyDecisionTreeClassifier":
        ...
        return self

    # X: (m, p); возвращает (m, n_classes_)
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return ...

    # X: (m, p); возвращает (m,) целые метки
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...

    # возвращает int: глубина дерева, у дерева из одного листа 0
    def get_depth(self) -> int:
        return ...

    # возвращает int: число листьев
    def get_n_leaves(self) -> int:
        return ...

In [ ]:
# --- проверка ---
stump = MyDecisionTreeClassifier(max_depth=1)
stump.fit(X_train, y_train)
sklearn_stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
sklearn_stump.fit(X_train, y_train)
assert np.allclose(stump.predict_proba(X_test), sklearn_stump.predict_proba(X_test)), "глубина 1: вероятности не совпали с sklearn"

# при равном приросте разбиения могут отличаться от sklearn, поэтому на глубине 2
# сравнивают не вероятности, а неоднородность листьев — она у лучших разбиений одна
for depth in [1, 2]:
    mine = MyDecisionTreeClassifier(max_depth=depth)
    mine.fit(X_train, y_train)

    reference = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    reference.fit(X_train, y_train)

    mine_leaf_impurity = sum(leaf["n_samples"] * leaf["impurity"] for leaf in leaves_of(mine.tree_)) / len(y_train)
    is_leaf = reference.tree_.children_left == -1
    leaf_sizes = reference.tree_.n_node_samples[is_leaf]
    reference_leaf_impurity = (leaf_sizes * reference.tree_.impurity[is_leaf]).sum() / len(y_train)

    assert np.isclose(mine_leaf_impurity, reference_leaf_impurity), f"глубина {depth}: неоднородность листьев больше, чем у sklearn, — разбиения не лучшие"
    assert mine.get_n_leaves() == reference.get_n_leaves(), f"глубина {depth}: число листьев не совпало с sklearn"
    assert mine.get_depth() == depth, f"глубина {depth}: get_depth вернул {mine.get_depth()}"

mine = MyDecisionTreeClassifier()
mine.fit(X_train, y_train)
probabilities = mine.predict_proba(X_test)
predictions = mine.predict(X_test)

assert mine.n_classes_ == 3, "n_classes_ — наибольшая метка плюс один"
assert probabilities.shape == (len(X_test), 3), "predict_proba возвращает (m, n_classes_)"
assert np.mean(mine.predict(X_train) == y_train) == 1.0, "дерево без ограничений не ошибается на обучающей выборке"
assert np.array_equal(predictions, probabilities.argmax(axis=1)), "predict — класс с наибольшей долей"

test_accuracy = np.mean(predictions == y_test)
print(f"без ограничений: глубина {mine.get_depth()}, листьев {mine.get_n_leaves()}, accuracy на тестовом сете {test_accuracy:.3f}")
print("проверки пройдены")

### Задача 7. Регрессор

Класс `MyDecisionTreeRegressor` — то же дерево для регрессии. Критерий —
дисперсия ответов узла $H(R) = \frac{1}{|R|}\sum_{i \in R}(y_i - \bar{y}_R)^2$, где $\bar{y}_R$ — среднее ответов
узла; прирост считается по той же формуле, значение узла `"value"` — `float`,
среднее ответов узла. Остальные ключи узла и правила остановки — как в задаче 5,
правила выбора при равенстве — как в задачах 3–4; условие «все объекты одного
класса» заменяется на «все ответы узла одинаковы». Можно добавить в `CRITERIA`
свой критерий и переиспользовать функции задач 2–4: аргумент `criterion` в них —
ключ словаря `CRITERIA`, а `n_classes` в регрессии не используется. Можно
написать и отдельные функции.

In [ ]:
class MyDecisionTreeRegressor:
    def __init__(self, max_depth: int | None = None, min_samples_split: int = 2, min_samples_leaf: int = 1):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf

    # X: (n, p), y: (n,) вещественные ответы; задает self.tree_, возвращает self
    def fit(self, X: np.ndarray, y: np.ndarray) -> "MyDecisionTreeRegressor":
        ...
        return self

    # X: (m, p); возвращает (m,): среднее ответов листа
    def predict(self, X: np.ndarray) -> np.ndarray:
        return ...

    # возвращает int: глубина дерева, у дерева из одного листа 0
    def get_depth(self) -> int:
        return ...

    # возвращает int: число листьев
    def get_n_leaves(self) -> int:
        return ...

In [ ]:
# --- проверка ---
for depth in [1, 2, 3]:
    mine = MyDecisionTreeRegressor(max_depth=depth)
    mine.fit(Xd_train, yd_train)

    reference = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    reference.fit(Xd_train, yd_train)

    assert np.allclose(mine.predict(Xd_test), reference.predict(Xd_test)), f"глубина {depth}: предсказания не совпали с sklearn"
    assert mine.get_n_leaves() == reference.get_n_leaves(), f"глубина {depth}: число листьев не совпало с sklearn"

mine = MyDecisionTreeRegressor(min_samples_leaf=20)
mine.fit(Xd_train, yd_train)
root = mine.tree_

assert isinstance(root["value"], float), "значение узла регрессии — float"
assert np.isclose(root["value"], yd_train.mean()), "значение корня — среднее ответов"
assert np.isclose(root["impurity"], yd_train.var()), "критерий корня — дисперсия ответов"

test_mse = np.mean((mine.predict(Xd_test) - yd_test) ** 2)
print(f"min_samples_leaf = 20: листьев {mine.get_n_leaves()}, среднеквадратичная ошибка (MSE) на тестовом сете {test_mse:.0f}")
print("проверки пройдены")

## Блок C. Эксперименты

### Задача 8. Свое дерево и sklearn

Для каждой глубины из `DEPTHS` обучите `MyDecisionTreeClassifier`
и `DecisionTreeClassifier(random_state=SEED)` на всех признаках вин
и запишите в таблицу `depth_table` accuracy на обучающей выборке и на тестовом
сете и время обучения в секундах (`time.perf_counter`). Индекс таблицы —
глубины из `DEPTHS` с именем `"глубина"`, столбцы — `DEPTH_COLUMNS`
в этом порядке. Затем нарисуйте accuracy на тестовом сете обеих реализаций
от глубины.

In [ ]:
DEPTHS = [1, 2, 3, 4, 5, 6, 8]
DEPTH_COLUMNS = ["обучение, своя", "тест, своя", "обучение, sklearn", "тест, sklearn", "время, своя", "время, sklearn"]

In [ ]:
depth_table: pd.DataFrame = ...     # (len(DEPTHS), 6): индекс "глубина", столбцы DEPTH_COLUMNS

In [ ]:
# --- проверка ---
assert list(depth_table.index) == DEPTHS, "индекс таблицы — глубины из DEPTHS"
assert depth_table.index.name == "глубина", "имя индекса — «глубина»"
assert list(depth_table.columns) == DEPTH_COLUMNS, "столбцы — DEPTH_COLUMNS в том же порядке"
for depth in [1, 3]:
    reference = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    reference.fit(X_train, y_train)
    expected = reference.score(X_test, y_test)
    assert np.isclose(depth_table.loc[depth, "тест, sklearn"], expected), f"глубина {depth}: accuracy sklearn не совпала с пересчетом"

assert np.isclose(depth_table.loc[1, "тест, своя"], depth_table.loc[1, "тест, sklearn"]), "при глубине 1 реализации должны совпасть"
assert (depth_table[["время, своя", "время, sklearn"]] > 0).all().all(), "время обучения должно быть положительным"
print(depth_table.round(4))
print("проверки пройдены")

**Вопрос.** Во сколько раз своя реализация медленнее sklearn и почему?
При глубине 3 и больше accuracy своей реализации на тестовом сете может
отличаться от sklearn, хотя оба дерева строит один жадный алгоритм
с одним критерием. Найдите узел, где деревья расходятся, и объясните
причину: что общего у двух выбранных разбиений и почему на обучающей
выборке деревья ошибаются одинаково.

*Ответ пишите здесь.*

### Задача 9. Важность признаков

Важность признака по среднему уменьшению неоднородности (mean decrease
in impurity, MDI) — сумма по всем
внутренним узлам, разбитым по этому признаку, величины
$n_t H(t) - n_{t_l} H(t_l) - n_{t_r} H(t_r)$, где $n_t$ — число объектов узла, $t_l$ и $t_r$ — потомки.
Вектор важностей нормируют на сумму 1; у дерева из одного листа все важности
равны нулю. Посчитайте `wine_importances` для `MyDecisionTreeClassifier(max_depth=3)`
на винах и нарисуйте рядом с `feature_importances_` дерева sklearn той же глубины.
Небольшие расхождения на графике возможны по той же причине, что в вопросе
после задачи 8.

In [ ]:
# node: корень дерева в формате из условия, n_features = p
# возвращает (p,): важности, нормированные на сумму 1
def tree_feature_importances(node: dict, n_features: int) -> np.ndarray:
    return ...


wine_tree: MyDecisionTreeClassifier = ...    # max_depth=3, обученный на X_train
wine_importances: np.ndarray = ...  # (13,)

In [ ]:
# --- проверка ---
stump = MyDecisionTreeClassifier(max_depth=1)
stump.fit(X_train, y_train)
sklearn_stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
sklearn_stump.fit(X_train, y_train)
stump_importances = tree_feature_importances(stump.tree_, 13)
assert np.allclose(stump_importances, sklearn_stump.feature_importances_), "глубина 1: MDI не совпала с sklearn"

leaf_only = {"value": np.array([1.0]), "n_samples": 5, "impurity": 0.0}
assert np.allclose(tree_feature_importances(leaf_only, 4), np.zeros(4)), "у дерева из одного листа важности нулевые"

assert wine_importances.shape == (13,), "wine_importances — 13 чисел"
assert np.allclose(wine_importances, tree_feature_importances(wine_tree.tree_, 13)), "wine_importances — MDI дерева wine_tree"
assert np.isclose(wine_importances.sum(), 1), "важности в сумме дают 1"

top_three = np.argsort(wine_importances)[::-1][:3]
print("три самых важных признака:", [FEATURE_NAMES[j] for j in top_three])
print("проверки пройдены")

## Бонусы

### Задача 10, бонус. Быстрый поиск порога

`best_threshold` из задачи 3 для каждого порога заново считает критерий
по всем объектам узла, это $O(n^2)$ операций. Напишите `best_threshold_fast`
с тем же контрактом и теми же правилами при равенстве, которая работает
за $O(n \log n)$; на $n = 30\,000$ объектов — меньше секунды. Как этого добиться,
описано в конспекте, раздел 3, «Вычисление лучшего порога».

In [ ]:
# тот же контракт, что у best_threshold, criterion: "gini" или "entropy"
def best_threshold_fast(x: np.ndarray, y: np.ndarray, criterion: str, n_classes: int,
                        min_samples_leaf: int = 1) -> tuple[float, float] | None:
    return ...

In [ ]:
# --- проверка ---
rng = np.random.default_rng(0)

for criterion in ["gini", "entropy"]:
    x_check = np.round(rng.normal(size=300), 1)            # с повторяющимися значениями
    y_check = rng.integers(0, 3, 300)

    slow_threshold, slow_gain = best_threshold(x_check, y_check, criterion, 3, 5)
    fast_threshold, fast_gain = best_threshold_fast(x_check, y_check, criterion, 3, 5)
    assert np.isclose(slow_gain, fast_gain), f"{criterion}: прирост не совпал с best_threshold"
    assert np.isclose(slow_threshold, fast_threshold), f"{criterion}: порог не совпал с best_threshold"

constant_x = np.full(10, 1.0)
random_labels = rng.integers(0, 2, 10)
assert best_threshold_fast(constant_x, random_labels, "gini", 2) is None, "у постоянного признака нет порогов"

x_big = rng.normal(size=30_000)
y_big = rng.integers(0, 3, 30_000)
started = time.perf_counter()
best_threshold_fast(x_big, y_big, "gini", 3)
elapsed = time.perf_counter() - started
assert elapsed < 1.0, f"на 30 000 объектов ушло {elapsed:.2f} с"
print(f"30 000 объектов: {elapsed * 1000:.0f} мс")
print("проверки пройдены")

### Задача 11, бонус. Отсечение по сложности

Для дерева классификации $R(t) = \frac{n_t}{n} H(t)$, где $n$ — число объектов в корне, $R(T)$ —
сумма $R$ по листьям поддерева $T$, $|T|$ — число его листьев. Для внутреннего узла $t$
с поддеревом $T_t$

$$\alpha_{\text{eff}}(t) = \frac{R(t) - R(T_t)}{|T_t| - 1}.$$

`prune_tree` возвращает новое дерево (исходное не меняется): пока у дерева
есть внутренние узлы и наименьшее $\alpha_{\text{eff}}$ среди них не больше `alpha`, узел
с наименьшим $\alpha_{\text{eff}}$ схлопывается — становится листом с теми же `value`,
`n_samples` и `impurity`. Это то же правило, что параметр `ccp_alpha` в sklearn.
Затем посчитайте `pruning_table`: для своего дерева без ограничений на винах
и каждого `alpha` из `ALPHAS` — число листьев и accuracy на тестовом сете.

In [ ]:
ALPHAS = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1]

In [ ]:
# node: корень дерева классификации, alpha >= 0, n_total — число объектов в корне
# возвращает новое дерево в том же формате
def prune_tree(node: dict, alpha: float, n_total: int) -> dict:
    return ...


pruning_table: pd.DataFrame = ...   # (len(ALPHAS), 2): индекс "alpha", столбцы "листьев" и "тест"

In [ ]:
# --- проверка ---
def sklearn_to_dict(model, node=0):
    """Дерево sklearn в формате узлов лабораторной."""
    structure = model.tree_
    leaf_value = structure.value[node, 0]
    converted = {
        "value": leaf_value / leaf_value.sum(),
        "n_samples": int(structure.n_node_samples[node]),
        "impurity": float(structure.impurity[node]),
    }

    left_child = structure.children_left[node]
    right_child = structure.children_right[node]
    if left_child != -1:
        n_node = structure.n_node_samples[node]
        n_left = structure.n_node_samples[left_child]
        n_right = structure.n_node_samples[right_child]
        impurity_after = (n_left * structure.impurity[left_child] + n_right * structure.impurity[right_child]) / n_node
        gain = float(structure.impurity[node] - impurity_after)

        converted.update(
            feature=int(structure.feature[node]),
            threshold=float(structure.threshold[node]),
            gain=gain,
            left=sklearn_to_dict(model, left_child),
            right=sklearn_to_dict(model, right_child),
        )
    return converted


full_sklearn = DecisionTreeClassifier(random_state=SEED)
full_sklearn.fit(X_train, y_train)
converted = sklearn_to_dict(full_sklearn)

path_alphas = full_sklearn.cost_complexity_pruning_path(X_train, y_train).ccp_alphas
middle_alphas = (path_alphas[1:] + path_alphas[:-1]) / 2         # середины между порогами, чтобы не попасть на границу

for alpha in middle_alphas:
    pruned = prune_tree(converted, alpha, len(y_train))

    reference = DecisionTreeClassifier(random_state=SEED, ccp_alpha=alpha)
    reference.fit(X_train, y_train)
    expected = reference.get_n_leaves()

    got = len(leaves_of(pruned))
    assert got == expected, f"alpha = {alpha:.4f}: листьев {got}, у sklearn {expected}"
assert len(leaves_of(converted)) == full_sklearn.get_n_leaves(), "исходное дерево не должно меняться"
assert list(pruning_table.index) == ALPHAS, "индекс pruning_table — ALPHAS"
assert list(pruning_table.columns) == ["листьев", "тест"], "столбцы pruning_table — «листьев» и «тест»"
assert pruning_table["листьев"].is_monotonic_decreasing, "с ростом alpha листьев не больше"
print(pruning_table)
print("проверки пройдены")

### Задача 12, бонус. Предсказание при пропущенных значениях

Если у объекта пропущен (`np.nan`) признак, по которому разбит узел, объект
спускается в оба поддерева, и вектор вероятностей узла — среднее векторов
поддеревьев с весами долей обучающих объектов в потомках:

$$a_t(x) = \frac{n_{t_l}}{n_t}\, a_{t_l}(x) + \frac{n_{t_r}}{n_t}\, a_{t_r}(x).$$

Если признак узла известен, объект идет в одного потомка, как обычно; в листе
$a_t(x)$ — доли классов листа. Напишите `predict_proba_missing` для дерева
в формате лабораторной, затем сравните на винах accuracy своего дерева
глубины 4 при 30% скрытых значений признаков `flavanoids`, `color_intensity`
и `proline` на тестовом сете для двух способов: спуск в оба поддерева
и заполнение пропусков медианой признака по обучающей выборке. Маска
пропусков — `missing_mask`, результаты — в словаре `missing_accuracy`
с ключами `"спуск в оба поддерева"` и `"медиана"`. Одна маска на 54 объектах дает
шумный результат; сравните его с экспериментом в конспекте (раздел 9),
где accuracy усреднена по 30 маскам.

In [ ]:
MISSING_FEATURES = [FEATURE_NAMES.index(name) for name in ["flavanoids", "color_intensity", "proline"]]
rng_missing = np.random.default_rng(SEED)

missing_mask = np.zeros_like(X_test, dtype=bool)                          # True — значение скрыто
missing_mask[:, MISSING_FEATURES] = rng_missing.random((len(X_test), 3)) < 0.3

X_test_missing = np.where(missing_mask, np.nan, X_test)

In [ ]:
# node: корень дерева классификации, X: (m, p), пропуски — np.nan
# возвращает (m, n_classes)
def predict_proba_missing(node: dict, X: np.ndarray) -> np.ndarray:
    return ...


missing_accuracy: dict[str, float] = ...   # {"спуск в оба поддерева": число, "медиана": число}

In [ ]:
# --- проверка ---
model_check = MyDecisionTreeClassifier(max_depth=4)
model_check.fit(X_train, y_train)

without_missing = predict_proba_missing(model_check.tree_, X_test)
assert np.allclose(without_missing, model_check.predict_proba(X_test)), "без пропусков — обычное предсказание"

everything_missing = np.full((1, 13), np.nan)
answer_for_empty = predict_proba_missing(model_check.tree_, everything_missing)[0]
root_shares = np.bincount(y_train) / len(y_train)
assert np.allclose(answer_for_empty, root_shares), "если пропущено все, ответ — доли классов в корне"

with_missing = predict_proba_missing(model_check.tree_, X_test_missing)
assert np.allclose(with_missing.sum(axis=1), 1), "вероятности в сумме дают 1"
assert set(missing_accuracy) == {"спуск в оба поддерева", "медиана"}, "ключи missing_accuracy — «спуск в оба поддерева» и «медиана»"
print({name: round(value, 3) for name, value in missing_accuracy.items()})
print("проверки пройдены")